<a href="https://colab.research.google.com/github/Dara0510/BorisovaDaria/blob/main/hw_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️‍♂️ Домашнее задание: Детективное агентство "Кластеризация"

**Легенда:** Мы перехватили архив с 5000 загадочных изображений (`lyubachuba/mystery_images_5k`). У нас нет никаких подписей и категорий.

**Ваша задача:** использовать алгоритмы машинного обучения (без учителя), чтобы найти скрытую структуру в этих данных, сгруппировать похожие картинки и сделать выводы.

Мы используем легкую нейросеть (ResNet-18), чтобы быстро векторизовать картинки на GPU. Ваша цель — применить методы кластеризации, **подобрать параметры** и навайбкодить (с помощью ИИ или самостоятельно) **красивые визуализации** для каждого метода.

In [ ]:
!pip install -q datasets torch torchvision umap-learn plotly seaborn scikit-learn hdbscan pandas

import torch
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

### 1. Загрузка данных и Векторизация (Feature Extraction)
Используем ResNet-18. Эта сеть вычисляет векторы размером 512 чисел для каждой картинки. Она работает очень быстро и дает отличный баланс скорости и качества.


### ⚠️ ВАЖНО: Включаем GPU (Видеокарту)!

Нейросети (даже такие быстрые, как ResNet-18) работают очень медленно на обычном процессоре (CPU). Если оставить всё как есть, векторизация займет 20-30 минут. На видеокарте (GPU) это займет меньше 1 минуты.

**Как включить GPU в Google Colab:**
1. В верхнем меню нажмите: **Среда выполнения** (Runtime).
2. Выберите: **Сменить среду выполнения** (Change runtime type).
3. В разделе "Аппаратный ускоритель" (Hardware accelerator) выберите **T4 GPU** (или любой другой доступный GPU).
4. Нажмите **Сохранить**.

*Проверка:* При запуске ячейки ниже должно напечататься `Используем устройство: cuda`. Если написано `cpu` — вы забыли включить GPU!

In [ ]:
print("Загрузка датасета...")
dataset = load_dataset("lyubachuba/mystery_images_5k", split="train")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# Загружаем ResNet-18 без последнего слоя классификации
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity() # Убираем слой классификации, оставляем эмбеддинги (512)
resnet = resnet.to(device).eval()

# Предобработка картинок для ResNet
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_features(batch):
    images = [transform(img.convert("RGB")) for img in batch["image"]]
    images_tensor = torch.stack(images).to(device)
    with torch.no_grad():
        embeddings = resnet(images_tensor).cpu().numpy()
    return {"embedding": embeddings}

print("Извлекаем признаки (векторизация)...")

dataset_emb = dataset.map(
    extract_features,
    batched=True,
    batch_size=64,
    desc="Векторизация (ResNet-18)"
)

X = np.array(dataset_emb['embedding'])
print(f"Готово! Размерность данных X: {X.shape}")

### 2. Снижение размерности (UMAP)
Алгоритмам плотности и нам для визуализации нужны данные меньшей размерности.

In [ ]:
import umap

# Сжимаем 512 -> 3 компоненты для 3D графиков и лучшей кластеризации плотностью
reducer = umap.UMAP(n_components=3, random_state=42)
X_umap = reducer.fit_transform(X)
print(f"Размерность X_umap: {X_umap.shape}")

---
## 🚀 Ваша часть задания (Сделайте выводы в текстовых ячейках под кодом!)

### Задание 1. K-Means
**Что нужно:**
1. Попробуйте разное количество кластеров (`n_clusters`). Посчитайте метрики (Silhouette, Davies-Bouldin). Выберите количество кластеров через метод локтя.
2. **Визуализация (Vibe-code):** Напишите код для интерактивного 3D Scatter plot через `plotly.express` по данным `X_umap`, где точки раскрашены в цвета кластеров.
3. Выведите сетку 3x3 случайных картинок для 2-3 интересных кластеров.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import plotly.express as px

# ВАШ КОД ЗДЕСЬ:
# 1. Обучить KMeans
# 2. Посчитать метрики
# 3. Визуализировать в 3D Plotly
# 4. Вывести картинки из кластеров


**Выводы по K-Means:**
*(Напишите здесь: Легко ли алгоритм нашел кластеры? Какое количество кластеров кажется оптимальным? Что общего у картинок внутри кластеров?)*

### Задание 2. Иерархическая кластеризация
**Что нужно:**
1. Попробуйте параметры `linkage` ('ward', 'average', 'complete').
2. **Визуализация (Vibe-code):** Постройте раскрашенную дендрограмму (используйте `scipy.cluster.hierarchy.dendrogram`), взяв случайную подвыборку в 150-200 картинок (иначе график превратится в кашу).
3. Постройте 2D Scatter plot (Seaborn), визуально сравнив кластеры от разных `linkage`.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import seaborn as sns

# ВАШ КОД ЗДЕСЬ:

**Выводы по Иерархической кластеризации:**
*(Напишите здесь: Какой метод linkage дал самые осмысленные формы кластеров на графике? Видно ли по дендрограмме, на сколько групп логично бить датасет? Как выглядит иерархия?)*

### Задание 3. DBSCAN
**Что нужно:**
1. Подберите `eps` и `min_samples` в цикле (попробуйте разные комбинации). Посчитайте % шума для каждой.
2. **Визуализация (Vibe-code):** Сделайте график (Matplotlib), показывающий точки-кластеры в цвете, а точки-шум серым цветом (`label == -1`).
3. Выведите галерею "выбросов" (самых странных картинок, которые алгоритм отнес к шуму). Выведите то, что принадлежит к одному кластеру

In [ ]:
from sklearn.cluster import DBSCAN

# ВАШ КОД ЗДЕСЬ:

**Выводы по DBSCAN:**
*(Напишите здесь: Насколько сложно было подобрать параметры? Почему картинки-выбросы оказались в шуме?)*

### Задание 4. HDBSCAN
**Что нужно:**
1. Примените алгоритм (попробуйте изменить `min_cluster_size`).
2. У HDBSCAN есть фича — уверенность в кластеризации (probabilities_).
3. **Визуализация (Vibe-code):** Нарисуйте 2D или 3D Scatter (Plotly), где прозрачность (`opacity`) или размер точки (`size`) зависит от того, насколько HDBSCAN уверен, что точка принадлежит кластеру.

In [ ]:
import hdbscan

# ВАШ КОД ЗДЕСЬ:

**Выводы по HDBSCAN:**
*(Напишите здесь: Легче ли он настраивается, чем обычный DBSCAN? Много ли точек получили низкую вероятность?)*

### 🏆 5. Итоговое расследование
Соберите все метрики (Silhouette, Davies-Bouldin, % шума) в единую `pandas` таблицу и раскрасьте её с помощью `DataFrame.style.background_gradient()`.

In [ ]:
import pandas as pd

# ВАШ КОД ЗДЕСЬ (Таблица сравнения):

### 🔍 Гранд-вывод:
1. Какой алгоритм оказался лучшим математически (по метрикам)?
2. Какой алгоритм оказался лучшим визуально (субъективно)?
3. **Что это за датасет?** Посмотрев на картинки, какую общую тему/жанр вы бы им присвоили?
4. На какие группы уместнее всего разделить датасет?